In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import torch
import torch.nn.functional as F

/Users/timbarvenov/Documents/uam/sem1/uczenie_glebokie_all/uczenie_glebokie/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
documents = [
    """Mechanizm self-attention w Transformerze pozwala każdemu tokenowi
    brać pod uwagę inne tokeny w sekwencji. Dzięki temu model może
    efektywnie uchwycić zależności długodystansowe.""",

    """Beam search to deterministyczna strategia dekodowania, która
    utrzymuje kilka najlepszych hipotez równolegle na każdym kroku
    generacji. Pozwala to znaleźć ciąg tokenów o wysokim łącznym
    prawdopodobieństwie.""",

    """Top-p sampling (nucleus sampling) to stochastyczna metoda wyboru
    kolejnego tokenu. Wybieramy najmniejszy zbiór tokenów o skumulowanym
    prawdopodobieństwie co najmniej p, a następnie losujemy z tego
    zbioru. Daje to bardziej zróżnicowane odpowiedzi.""",

    """Modele Transformer składają się z warstw self-attention oraz
    warstw feed-forward. Dzięki równoległemu przetwarzaniu sekwencji
    są bardziej efektywne obliczeniowo niż modele RNN.""",

    """Retrieval-Augmented Generation (RAG) łączy mechanizm wyszukiwania
    dokumentów z generacją tekstu przy użyciu dużych modeli językowych.
    Najpierw wyszukiwane są relewantne fragmenty wiedzy, a następnie
    model generatywny używa ich jako kontekstu do odpowiedzi."""
]

In [3]:
queries = [
    "Czym różni się beam search od top-p sampling?",
    "Na czym polega mechanizm self-attention w Transformerze?",
    "Co to jest model RAG i jakie ma zalety?",
]

In [4]:
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = SentenceTransformer(embed_model_name)

In [19]:
gen_model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_model_name)

In [6]:
device = torch.device("mps")

In [9]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
gen_model.config.pad_token_id = tokenizer.pad_token_id

In [11]:
gen_model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [12]:
doc_embeddings = embed_model.encode(documents, convert_to_tensor=True)
doc_embeddings = F.normalize(doc_embeddings, p=2, dim=1)

In [13]:
def retrieve_top_k(query_text, k=2):
    q_emb = embed_model.encode([query_text], convert_to_tensor=True)
    q_emb = F.normalize(q_emb, p=2, dim=1)

    # cos similarity
    scores = torch.matmul(q_emb, doc_embeddings.T).squeeze(0)
    topk_scores, topk_indices = torch.topk(scores, k)

    retrieved_docs = [documents[i] for i in topk_indices.tolist()]
    return retrieved_docs, topk_scores.tolist()


def generate_answer(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip()

In [17]:
for q in queries[:1]:
    print(f"ZAPYTANIE: {q}\n")

    # Z RAG:
    retrieved_docs, scores = retrieve_top_k(q, k=2)

    context = ""
    for i, (doc, sc) in enumerate(zip(retrieved_docs, scores), start=1):
        context += f"[DOKUMENT {i} (score={sc:.3f})]\n{doc}\n\n"

    rag_prompt = (
        "Na podstawie poniższych dokumentów odpowiedz po polsku na pytanie.\n\n"
        f"{context}"
        f"Pytanie: {q}\n"
    )

    rag_answer = generate_answer(rag_prompt)

    # Bez RAG:
    vanilla_prompt = (
        "Odpowiedz po polsku na pytanie.\n\n"
        f"Pytanie: {q}\n"
        "Odpowiedź:"
    )

    no_rag_answer = generate_answer(vanilla_prompt)

    # ---- 6.3. Wynik ----
    print(">>> Odpowiedź Z RAG (z kontekstem dokumentów):")
    print(rag_answer, "\n")

    print(">>> Odpowiedź BEZ RAG (sam model):")
    print(no_rag_answer, "\n")

ZAPYTANIE: Czym różni się beam search od top-p sampling?

>>> Odpowiedź Z RAG (z kontekstem dokumentów):
[DOKUMENT 3 (score=0.542)]
Konor: Daje to bardziej zwzych
Dakty: Daje to bardziej zwzych
Dakty: Daje to bardziej zwzych
Dakty: Daje to bardziej zw 

>>> Odpowiedź BEZ RAG (sam model):
Polka: Czym różni się beam search od top-p sampling?
Odpowiedź:
Polka: Czym różni się beam search od top-p sampling?
Odpowiedź:
Polka: Czym różni się beam search od 

